<a href="https://colab.research.google.com/github/younguk072023/Thermal-Imaging/blob/main/real_yolo5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 열영상 기반 근육 자극 부하 영역 탐지

운동 중 촬영한 열영상에서 관심 영역을 탐지하고, 동일한 데이터 분할과 학습 조건에서 YOLOv5s·m·l·x를 비교하는 Colab 노트북입니다. 박스 클래스는 **CriticalHeat**와 **ModerateHeat** 두 가지입니다.

이 저장소에는 데이터와 학습 가중치가 없으므로 결과 수치는 저장되어 있지 않습니다. 아래 셀을 실행하면 실제 정답 라벨을 기준으로 Precision, Recall, mAP@0.5, mAP@0.5:0.95를 계산합니다.

## 1. 환경 준비

Colab에서 GPU 런타임을 선택하고, Google Drive의 최상위 폴더에 `data.zip`을 올려주세요. YOLOv5는 공식 저장소에서 가져오며, 실행 시 사용한 Git 커밋을 출력합니다. 원래 논문 실험의 YOLOv5 커밋은 확인되지 않아 논문 수치의 동일 재현을 보장하지 않습니다.

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

YOLOV5_DIR = Path('/content/yolov5')
if not YOLOV5_DIR.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/ultralytics/yolov5.git', str(YOLOV5_DIR)],
        check=True,
    )
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(YOLOV5_DIR / 'requirements.txt')],
    check=True,
)
commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=YOLOV5_DIR, text=True
).strip()
print('YOLOv5 commit:', commit)

## 2. 데이터 확인

압축파일은 아래 구조를 가정합니다. 이미지와 YOLO 형식의 박스 라벨은 train·val·test로 나뉘어 있어야 합니다.

```text
data/
  images/{train,val,test}/
  labels/{train,val,test}/
```

논문에서 사용한 증강 후 이미지 수는 총 525장(471/27/27)입니다. 셀은 실제 압축파일의 분할 수와 라벨 ID를 검사합니다. 라벨 ID가 15·16만 사용된 경우, 원본을 보존하고 별도의 작업용 사본에서 0·1로 바꿉니다. 그 외의 ID 조합은 자동으로 추측하지 않고 중단합니다.

In [ ]:
import zipfile

ARCHIVE = Path('/content/drive/MyDrive/data.zip')
EXTRACT_DIR = Path('/content/dataset')
RAW_DATA_DIR = EXTRACT_DIR / 'data'

if not ARCHIVE.is_file():
    raise FileNotFoundError(f'Google Drive에서 {ARCHIVE} 파일을 찾을 수 없습니다.')
if not RAW_DATA_DIR.exists():
    with zipfile.ZipFile(ARCHIVE) as archive:
        archive.extractall(EXTRACT_DIR)
if not RAW_DATA_DIR.is_dir():
    raise FileNotFoundError(
        f'{RAW_DATA_DIR} 폴더가 없습니다. data.zip 내부 구조를 확인하세요.'
    )
print('원본 데이터:', RAW_DATA_DIR)

In [ ]:
from collections import Counter
from pathlib import Path
import math
import shutil
import tempfile

SPLITS = ('train', 'val', 'test')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
EXPECTED_IMAGES = {'train': 471, 'val': 27, 'test': 27}
image_counts = {}
label_counts = {}
class_counts = {}
observed_ids = set()

for split in SPLITS:
    image_dir = RAW_DATA_DIR / 'images' / split
    label_dir = RAW_DATA_DIR / 'labels' / split
    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(f'{split}의 images/labels 폴더를 확인하세요.')

    images = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
    labels = sorted(label_dir.glob('*.txt'))
    image_stems = [p.stem for p in images]
    if len(set(image_stems)) != len(image_stems):
        raise ValueError(f'{split}에 확장자만 다른 중복 이미지 이름이 있습니다.')
    orphan_labels = {p.stem for p in labels} - set(image_stems)
    if orphan_labels:
        raise ValueError(f'{split}의 이미지와 짝이 없는 라벨: {sorted(orphan_labels)[:5]}')
    if not images or not labels:
        raise ValueError(f'{split}에 이미지 또는 정답 라벨이 없습니다.')

    counts = Counter()
    for label_path in labels:
        for line_number, line in enumerate(label_path.read_text().splitlines(), 1):
            parts = line.split()
            if not parts:
                continue
            if len(parts) != 5:
                raise ValueError(f'{label_path}:{line_number}은 YOLO 박스 형식(5개 값)이 아닙니다.')
            try:
                class_id = int(parts[0])
                box = [float(value) for value in parts[1:]]
            except ValueError as error:
                raise ValueError(f'{label_path}:{line_number}의 라벨 값을 확인하세요.') from error
            if any(not math.isfinite(value) for value in box):
                raise ValueError(f'{label_path}:{line_number}에 유효하지 않은 좌표가 있습니다.')
            if any(value < 0 or value > 1 for value in box) or box[2] <= 0 or box[3] <= 0:
                raise ValueError(f'{label_path}:{line_number}의 YOLO 좌표 범위를 확인하세요.')
            counts[class_id] += 1
            observed_ids.add(class_id)

    if not counts:
        raise ValueError(f'{split}의 라벨에 탐지 박스가 없습니다.')
    image_counts[split] = len(images)
    label_counts[split] = len(labels)
    class_counts[split] = dict(counts)

print('이미지 수:', image_counts)
print('라벨 파일 수:', label_counts)
print('분할별 박스 클래스 ID:', class_counts)
if image_counts != EXPECTED_IMAGES:
    print('주의: 실제 이미지 수가 논문의 471/27/27과 다릅니다. 평가 조건을 확인하세요.')

if observed_ids not in ({0, 1}, {15, 16}):
    raise ValueError(
        f'라벨 ID {sorted(observed_ids)}을 확인하세요. '
        '기대하는 ID 조합은 {0,1} 또는 {15,16}입니다. '
        '원본 라벨 정의를 확인한 뒤 수동으로 정리해야 합니다.'
    )

DATA_DIR = Path(tempfile.mkdtemp(prefix='thermal_yolo_')) / 'data'
shutil.copytree(RAW_DATA_DIR, DATA_DIR)
for cache_path in DATA_DIR.rglob('*.cache'):
    cache_path.unlink()
if observed_ids == {15, 16}:
    id_map = {15: 0, 16: 1}
    for split in SPLITS:
        for label_path in (DATA_DIR / 'labels' / split).glob('*.txt'):
            remapped = []
            for line in label_path.read_text().splitlines():
                parts = line.split()
                if parts:
                    parts[0] = str(id_map[int(parts[0])])
                    remapped.append(' '.join(parts))
            label_path.write_text('\n'.join(remapped) + ('\n' if remapped else ''))
    print('원본은 보존하고 작업용 사본의 라벨 ID 15/16을 0/1로 변환했습니다.')
    print('작업용 데이터:', DATA_DIR)
else:
    print('원본을 보존한 작업용 사본에서 라벨 ID 0/1을 그대로 사용합니다.')
    print('0=CriticalHeat, 1=ModerateHeat 순서가 원본 라벨 정의와 같은지 확인하세요.')

In [ ]:
import yaml

DATA_YAML = Path('/content/thermal_data.yaml')
config = {
    'train': str(DATA_DIR / 'images' / 'train'),
    'val': str(DATA_DIR / 'images' / 'val'),
    'test': str(DATA_DIR / 'images' / 'test'),
    'nc': 2,
    'names': ['CriticalHeat', 'ModerateHeat'],
}
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))
print(DATA_YAML.read_text())

## 3. YOLOv5s·m·l·x 학습

네 모델에 동일한 이미지 크기(256), 배치(16), 학습 횟수(100), 시드(42)를 적용합니다. 각 모델의 결과는 독립된 폴더에 저장합니다. GPU 메모리가 부족하면 모든 모델에 동일한 더 작은 배치를 지정한 후 새로 실행하세요.

In [ ]:
from datetime import datetime
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU가 감지되지 않았습니다. Colab 런타임에서 GPU를 선택하세요.')

MODELS = ('s', 'm', 'l', 'x')
IMAGE_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 100
SEED = 42
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUTPUT_DIR = Path('/content/thermal_runs') / RUN_ID
TRAIN_DIR = OUTPUT_DIR / 'train'
EVAL_DIR = OUTPUT_DIR / 'eval'
print('실험 결과 폴더:', OUTPUT_DIR)

In [ ]:
for size in MODELS:
    name = f'yolov5{size}'
    command = [
        sys.executable, 'train.py',
        '--imgsz', str(IMAGE_SIZE),
        '--batch-size', str(BATCH_SIZE),
        '--epochs', str(EPOCHS),
        '--data', str(DATA_YAML),
        '--weights', f'{name}.pt',
        '--seed', str(SEED),
        '--project', str(TRAIN_DIR),
        '--name', name,
    ]
    print(f'학습 시작: {name}', flush=True)
    subprocess.run(command, cwd=YOLOV5_DIR, check=True)
    best_weights = TRAIN_DIR / name / 'weights' / 'best.pt'
    if not best_weights.is_file():
        raise FileNotFoundError(f'{name}의 best.pt가 생성되지 않았습니다: {best_weights}')
    print('최적 가중치:', best_weights)

## 4. 검증셋 선택과 테스트셋 평가

먼저 validation 분할의 mAP@0.5로 시각화할 모델을 선택합니다. 그 뒤 네 모델을 모두 별도의 test 분할에서 평가합니다. Precision, Recall, mAP@0.5, mAP@0.5:0.95는 YOLOv5 검증 함수가 정답 박스와 비교해 계산하며, 예측 confidence로 대신하지 않습니다.

In [ ]:
import os
import pandas as pd
from IPython.display import display

os.chdir(YOLOV5_DIR)
if str(YOLOV5_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOV5_DIR))
from val import run as validate

def evaluate_split(split):
    rows = []
    for size in MODELS:
        name = f'yolov5{size}'
        weights = TRAIN_DIR / name / 'weights' / 'best.pt'
        if not weights.is_file():
            raise FileNotFoundError(f'{weights}가 없습니다. 학습 셀을 먼저 실행하세요.')
        metrics, _, _ = validate(
            data=str(DATA_YAML),
            weights=str(weights),
            imgsz=IMAGE_SIZE,
            batch_size=BATCH_SIZE,
            task=split,
            project=str(EVAL_DIR / split),
            name=name,
            plots=True,
        )
        rows.append({
            'model': name,
            'precision': float(metrics[0]),
            'recall': float(metrics[1]),
            'mAP50': float(metrics[2]),
            'mAP50-95': float(metrics[3]),
        })
    return pd.DataFrame(rows)

validation_comparison = evaluate_split('val')
test_comparison = evaluate_split('test')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
validation_comparison.to_csv(OUTPUT_DIR / 'validation_comparison.csv', index=False)
test_comparison.to_csv(OUTPUT_DIR / 'test_comparison.csv', index=False)

print('Validation 결과 (모델 선택용)')
display(validation_comparison.round(3))
print('Test 결과 (별도 평가)')
display(test_comparison.round(3))

In [ ]:
import matplotlib.pyplot as plt

ax = test_comparison.set_index('model')[
    ['precision', 'recall', 'mAP50', 'mAP50-95']
].plot(kind='bar', figsize=(10, 5), ylim=(0, 1), rot=0)
ax.set_title('YOLOv5 test-set comparison')
ax.set_ylabel('Score')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'test_comparison.png', dpi=180)
plt.show()

## 5. 탐지 결과 확인

시각화에는 validation 분할에서 mAP@0.5가 가장 높은 모델을 사용합니다. 테스트 이미지의 예측 상자를 확인할 수 있지만, 상자에 표시된 confidence는 개별 예측 점수이며 평가 지표가 아닙니다.

In [ ]:
from IPython.display import Image, display

best_name = validation_comparison.loc[validation_comparison['mAP50'].idxmax(), 'model']
best_weights = TRAIN_DIR / best_name / 'weights' / 'best.pt'
DETECT_DIR = OUTPUT_DIR / 'detect'
subprocess.run(
    [
        sys.executable, 'detect.py',
        '--weights', str(best_weights),
        '--imgsz', str(IMAGE_SIZE),
        '--conf-thres', '0.25',
        '--source', str(DATA_DIR / 'images' / 'test'),
        '--project', str(DETECT_DIR),
        '--name', best_name,
    ],
    cwd=YOLOV5_DIR,
    check=True,
)
predictions = sorted(
    path for path in (DETECT_DIR / best_name).iterdir()
    if path.suffix.lower() in IMAGE_EXTENSIONS
)
print('시각화 모델:', best_name)
for image_path in predictions[:4]:
    display(Image(filename=str(image_path), width=450))

## 해석 범위

여기서 비교하는 것은 열영상의 **부하 관련 영역 탐지 성능**입니다. 이를 근육 상태나 질병의 진단 정확도로 해석할 수 없습니다. 데이터가 적고 증강 이미지가 포함되어 있으므로, 원본 이미지 단위의 분리 여부와 다른 촬영 조건에서의 일반화 성능을 추가로 확인해야 합니다.